# TPAE II — Entrega 5 | Aquisição, limpeza e tratamento de dados

Equipe 06 — Thiago Ramirez Balotta Starikoff, João Marcos Nery da Silva e Cauã Antero do Nascimento.

**Objetivo:** integrar população urbana/rural, alfabetização e renda per capita dos 223 municípios da Paraíba, com dados do Censo 2022/IBGE. Fontes: agregados por setores, tabela 2.1, tabelas 9543 e 10295. Arquivo final: `base_municipal_pb_tratada.csv`.


## 1. Aquisição reproduzível pelo GitHub
Os quatro arquivos originais são lidos do repositório público, sem upload manual.

In [ ]:
import os
import subprocess
import zipfile
import pandas as pd

repositorio = "https://github.com/thiago-starikoff-academico/TPAE-2-projeto.git"
pasta = "/content/TPAE-2-projeto"

if os.path.isdir(os.path.join(pasta, ".git")):
    subprocess.run(["git", "-C", pasta, "pull"], check=True)
else:
    subprocess.run(["git", "clone", repositorio, pasta], check=True)

pasta_raw = os.path.join(pasta, "data", "raw")
pasta_extraida = "/content/dados_ibge"
os.makedirs(pasta_extraida, exist_ok=True)

for nome in [
    "Agregados_por_setores_basico_BR_20260520.zip",
    "2_1_Populacao_residente_Area_territorial_Densidade_demografica_xlsx.zip"
]:
    with zipfile.ZipFile(os.path.join(pasta_raw, nome)) as arquivo:
        arquivo.extractall(pasta_extraida)

arquivo_setores = os.path.join(pasta_extraida, "Agregados_por_setores_basico_BR.csv")
arquivo_populacao = os.path.join(pasta_extraida, "tabela_2_1_PB.xlsx")
arquivo_alfabetizacao = os.path.join(pasta, "Tabela 9543.xlsx")
arquivo_renda = os.path.join(pasta, "Tabela 10295.xlsx")

for caminho in [arquivo_setores, arquivo_populacao, arquivo_alfabetizacao, arquivo_renda]:
    assert os.path.isfile(caminho), f"Arquivo não encontrado: {caminho}"
print("Quatro arquivos originais encontrados e extraídos.")


## 2. Diagnóstico e tratamento dos setores censitários

In [ ]:
import pandas as pd

caminho = "/content/dados_ibge/Agregados_por_setores_basico_BR.csv"

setores = pd.read_csv(
    caminho,
    sep=";",
    encoding="latin1",
    low_memory=False
)

print("Dimensões:", setores.shape)
print("Colunas:", setores.columns.tolist()[:15])

In [ ]:
setores_pb = setores[setores["CD_UF"] == 25].copy()

print("Setores da Paraíba:", len(setores_pb))
print("Municípios:", setores_pb["CD_MUN"].nunique())
print("\nSituação dos setores:")
print(setores_pb["SITUACAO"].value_counts(dropna=False))

print("\nCódigos de setores duplicados:",
      setores_pb["CD_SETOR"].duplicated().sum())

In [ ]:
setores_sem_situacao = setores_pb[
    setores_pb["SITUACAO"].isna()
]

print(setores_sem_situacao[
    ["CD_SETOR", "CD_MUN", "NM_MUN", "SITUACAO", "v0001"]
].to_string(index=False))

In [ ]:
setores_pb_tratados = setores_pb[
    setores_pb["SITUACAO"].isin(["Urbana", "Rural"])
].copy()

print("Registros antes:", len(setores_pb))
print("Registros depois:", len(setores_pb_tratados))
print("Registros excluídos:", len(setores_pb) - len(setores_pb_tratados))

print(
    "População antes:",
    setores_pb["v0001"].sum()
)

print(
    "População depois:",
    setores_pb_tratados["v0001"].sum()
)

In [ ]:
pop_situacao = (
    setores_pb_tratados
    .groupby(["CD_MUN", "NM_MUN", "SITUACAO"])["v0001"]
    .sum()
    .reset_index()
)

print("Municípios:", pop_situacao["CD_MUN"].nunique())
print("População agregada:", pop_situacao["v0001"].sum())

print("\nPrimeiras linhas:")
print(pop_situacao.head(10))

## 3. Conferência com a população oficial do IBGE
A comparação serve como controle de qualidade, não como substituição dos valores setoriais.

In [ ]:
tabela_ibge = pd.read_excel(arquivo_populacao, header=None)
pop_oficial = tabela_ibge.iloc[5:, :2].copy()
pop_oficial.columns = ["municipio", "populacao_oficial"]
pop_oficial["municipio"] = (pop_oficial["municipio"].astype(str)
    .str.replace(r"\s*\(PB\)$", "", regex=True).str.strip())
pop_oficial["populacao_oficial"] = pd.to_numeric(
    pop_oficial["populacao_oficial"], errors="coerce")
pop_oficial = pop_oficial.dropna(subset=["populacao_oficial"]).copy()

pop_setores = (pop_situacao.groupby("NM_MUN", as_index=False)["v0001"].sum()
    .rename(columns={"NM_MUN": "municipio", "v0001": "populacao_setores"}))
comparacao = pop_oficial.merge(pop_setores, on="municipio", how="outer", indicator=True,
                              validate="one_to_one")
comparacao["diferenca"] = comparacao["populacao_setores"] - comparacao["populacao_oficial"]
print("Municípios comparados:", len(comparacao))
print("Sem correspondência:", (comparacao["_merge"] != "both").sum())
print("Com diferença:", comparacao["diferenca"].fillna(0).ne(0).sum())
print("Diferença total:", comparacao["diferenca"].sum())
print(comparacao.loc[comparacao["diferenca"].fillna(0).ne(0),
                    ["municipio", "populacao_oficial", "populacao_setores", "diferenca"]])


## 4. Consolidação municipal

In [ ]:
pop_municipios = pop_situacao.pivot(
    index=["CD_MUN", "NM_MUN"],
    columns="SITUACAO",
    values="v0001"
).reset_index()

pop_municipios.columns.name = None

pop_municipios = pop_municipios.rename(columns={
    "CD_MUN": "codigo_ibge",
    "NM_MUN": "municipio",
    "Rural": "populacao_rural",
    "Urbana": "populacao_urbana"
})

pop_municipios["populacao_total"] = (
    pop_municipios["populacao_rural"] +
    pop_municipios["populacao_urbana"]
)

pop_municipios["percentual_rural"] = (
    pop_municipios["populacao_rural"] /
    pop_municipios["populacao_total"] * 100
).round(2)

print("Dimensões:", pop_municipios.shape)
print("População total:", pop_municipios["populacao_total"].sum())
print("\nPrimeiras linhas:")
print(pop_municipios.head())

## 5. Alfabetização e integração

In [ ]:
alfabetizacao_bruta = pd.read_excel(arquivo_alfabetizacao, header=None)
print("Dimensões:", alfabetizacao_bruta.shape)


In [ ]:
alfabetizacao = alfabetizacao_bruta.iloc[9:, :2].copy()

alfabetizacao.columns = [
    "municipio",
    "taxa_alfabetizacao"
]

alfabetizacao["municipio"] = (
    alfabetizacao["municipio"]
    .astype(str)
    .str.replace(r"\s*\(PB\)$", "", regex=True)
    .str.strip()
)

alfabetizacao["taxa_alfabetizacao"] = pd.to_numeric(
    alfabetizacao["taxa_alfabetizacao"],
    errors="coerce"
)

# Manter apenas municípios presentes na base principal
alfabetizacao = alfabetizacao[
    alfabetizacao["municipio"].isin(pop_municipios["municipio"])
].copy()

print("Municípios:", len(alfabetizacao))
print("Valores ausentes:", alfabetizacao["taxa_alfabetizacao"].isna().sum())
print("Municípios duplicados:", alfabetizacao["municipio"].duplicated().sum())

print("\nPrimeiras linhas:")
print(alfabetizacao.head())

In [ ]:
# Conferir correspondência entre as bases
faltantes = set(pop_municipios["municipio"]) - set(alfabetizacao["municipio"])

print("Municípios sem alfabetização:", len(faltantes))
print("Taxas fora do intervalo 0 a 100:",
      (~alfabetizacao["taxa_alfabetizacao"].between(0, 100)).sum())

# Integrar os dados
base_municipal = pop_municipios.merge(
    alfabetizacao,
    on="municipio",
    how="left",
    validate="one_to_one"
)

print("\nDimensões da base:", base_municipal.shape)
print("Alfabetização ausente após integração:",
      base_municipal["taxa_alfabetizacao"].isna().sum())

print("\nPrimeiras linhas:")
print(base_municipal.head())

## 6. Renda per capita e integração

In [ ]:
renda_bruta = pd.read_excel(arquivo_renda, header=None)
print("Dimensões:", renda_bruta.shape)


In [ ]:
renda = renda_bruta.iloc[8:, :2].copy()

renda.columns = [
    "municipio",
    "renda_per_capita"
]

renda["municipio"] = (
    renda["municipio"]
    .astype(str)
    .str.replace(r"\s*\(PB\)$", "", regex=True)
    .str.strip()
)

renda["renda_per_capita"] = pd.to_numeric(
    renda["renda_per_capita"],
    errors="coerce"
)

# Manter apenas municípios da base principal
renda = renda[
    renda["municipio"].isin(base_municipal["municipio"])
].copy()

print("Municípios:", len(renda))
print("Valores ausentes:", renda["renda_per_capita"].isna().sum())
print("Municípios duplicados:", renda["municipio"].duplicated().sum())

print("\nPrimeiras linhas:")
print(renda.head())

In [ ]:
# Verificar correspondência entre os municípios
faltantes_renda = (
    set(base_municipal["municipio"]) -
    set(renda["municipio"])
)

print("Municípios sem renda:", len(faltantes_renda))
print("Rendas negativas:", (renda["renda_per_capita"] < 0).sum())

# Integrar renda à base municipal
base_municipal = base_municipal.merge(
    renda,
    on="municipio",
    how="left",
    validate="one_to_one"
)

print("\nDimensões da base final:", base_municipal.shape)
print(
    "Rendas ausentes após integração:",
    base_municipal["renda_per_capita"].isna().sum()
)

print("\nPrimeiras linhas:")
print(base_municipal.head())

## 7. Testes de integridade

In [ ]:
print("Dimensões:", base_municipal.shape)

print("\nValores ausentes por coluna:")
print(base_municipal.isna().sum())

print("\nMunicípios duplicados:",
      base_municipal["municipio"].duplicated().sum())

print("Códigos IBGE duplicados:",
      base_municipal["codigo_ibge"].duplicated().sum())

print("Populações negativas:",
      (base_municipal[
          ["populacao_rural", "populacao_urbana", "populacao_total"]
      ] < 0).sum().sum())

print("Totais populacionais inconsistentes:",
      (
          base_municipal["populacao_rural"] +
          base_municipal["populacao_urbana"] !=
          base_municipal["populacao_total"]
      ).sum())

print("Percentuais rurais inválidos:",
      (~base_municipal["percentual_rural"].between(0, 100)).sum())

print("Taxas de alfabetização inválidas:",
      (~base_municipal["taxa_alfabetizacao"].between(0, 100)).sum())

print("Rendas negativas:",
      (base_municipal["renda_per_capita"] < 0).sum())

print("\nPopulação total:",
      base_municipal["populacao_total"].sum())

In [ ]:
assert len(base_municipal) == 223
assert base_municipal.shape[1] == 8
assert not base_municipal.isna().any().any()
assert not base_municipal["codigo_ibge"].duplicated().any()
assert not base_municipal["municipio"].duplicated().any()
assert (base_municipal["populacao_rural"] + base_municipal["populacao_urbana"] == base_municipal["populacao_total"]).all()
assert base_municipal["percentual_rural"].between(0, 100).all()
assert base_municipal["taxa_alfabetizacao"].between(0, 100).all()
assert (base_municipal["renda_per_capita"] >= 0).all()
assert (base_municipal[["populacao_rural", "populacao_urbana", "populacao_total"]] >= 0).all().all()
assert (comparacao["_merge"] == "both").all()
print("Todos os testes de integridade passaram.")


## 8. Exportação e resumo final
O arquivo CSV tratado é salvo em `/content` e pode ser baixado pelo painel de arquivos do Colab.

In [ ]:
base_municipal.to_csv(
    "/content/base_municipal_pb_tratada.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo exportado com sucesso!")
print("Dimensões:", base_municipal.shape)

### Limitações documentadas
- Cinco setores da Paraíba sem classificação urbana/rural foram excluídos; todos possuíam população zero.
- A soma populacional setorial é 3.974.687, enquanto a tabela oficial registra 3.974.495: diferença de 192 habitantes, localizada em Alhandra (17) e Jacaraú (175). Os valores foram preservados e a divergência deve constar no relatório de qualidade.
- Alfabetização: percentual da população com 15 anos ou mais. Renda: rendimento nominal mensal domiciliar per capita em reais, referente a 2022. Não são medidas intercambiáveis.
